# 041 — Random Forest, boosting y ensembles

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Principio:** la varianza del promedio de B modelos con varianza σ² y correlación ρ es
`ρσ² + (1−ρ)σ²/B` — promediar ayuda hasta el techo que fija la correlación: la clave es
la **diversidad**.

**Random forest (paralelo, baja varianza):** B árboles profundos sobre muestras bootstrap
(~63 % de los datos cada una) + `m` features aleatorias por nodo para descorrelacionar.
Error OOB como validación gratis. Más árboles no sobreajustan, solo saturan.

**Boosting (secuencial, baja sesgo):** modelo aditivo `F_M = F₀ + Σ ν·h_m` de árboles
pequeños. AdaBoost re-pondera los ejemplos fallados (`α = ½·ln((1−err)/err)`);
gradient boosting ajusta cada árbol a los pseudo-residuos de una pérdida diferenciable.
Sí sobreajusta con M grande: early stopping en validación, learning rate ν pequeño.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** P(los 3) = 0.8·0.7·0.6 = 0.336. Exactamente 2:
0.8·0.7·0.4 = 0.224; 0.8·0.3·0.6 = 0.144; 0.2·0.7·0.6 = 0.084 → suma 0.452.
**P(mayoría) = 0.336 + 0.452 = 0.788** — apenas por debajo del mejor individual (0.8):
con miembros desparejos, el voto simple puede no superar al mejor. Si los tres cometieran
los mismos errores (ρ=1), la mayoría acertaría exactamente cuando acierta el patrón común:
el ensemble no añade nada.

**Ejercicio 2.** (a) (1−1/1000)¹⁰⁰⁰ ≈ 0.3677 ≈ e⁻¹. (b) Cada árbol deja fuera al ejemplo
con probabilidad 0.368 → esperados 200·0.368 ≈ **74 árboles** votan en su predicción OOB.

**Ejercicio 3.** err₁ = 2/8 = 0.25. α₁ = ½·ln(0.75/0.25) = ½·ln 3 ≈ 0.549.
Sin normalizar: fallados 0.125·e^0.549 ≈ 0.2165; acertados 0.125·e^−0.549 ≈ 0.0722.
Suma = 2·0.2165 + 6·0.0722 = 0.4330 + 0.4330 = 0.866. Normalizados: fallados
0.2165/0.866 = **0.25** cada uno; acertados 0.0722/0.866 ≈ **0.0833** cada uno.
Suma: 2·0.25 + 6·0.0833 = 1 ✓. (AdaBoost siempre deja el peso total de los fallados
en ½.)

**Ejercicio 4.** Elegir el mejor stump usa un solo corte; el comité usa 5 y su frontera
efectiva es "positivo si x supera al umbral mediano" (mayoría de 5 stumps del mismo signo
= el umbral mediano decide). Para la accuracy exacta del comité harían falta las
**predicciones por ejemplo** de cada stump (o los datos), no solo la accuracy agregada:
la ganancia del ensemble depende de la estructura conjunta de los errores, información
que las accuracies individuales no contienen.


In [ ]:
result = run_lab("ml", seed=41)
assert result["kind"] == "ml"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — mayoría de 3 clasificadores independientes
p1, p2, p3 = 0.8, 0.7, 0.6
p_mayoria = (p1 * p2 * p3
             + p1 * p2 * (1 - p3)
             + p1 * (1 - p2) * p3
             + (1 - p1) * p2 * p3)
print(f"P(mayoría acierta) = {p_mayoria:.3f}  vs mejor individual = {max(p1, p2, p3)}")
# Con miembros desparejos e independientes la mayoría simple no garantiza superar al mejor.


In [ ]:
# Ejercicios 2 y 3 — OOB y una ronda de AdaBoost
import math

fuera = (1 - 1 / 1000) ** 1000
print(f"fracción OOB ≈ {fuera:.4f}  (e^-1 = {math.exp(-1):.4f})")
print(f"árboles OOB esperados de 200: {200 * fuera:.0f}")

n, fallados = 8, {3, 7}
err1 = len(fallados) / n
alpha1 = 0.5 * math.log((1 - err1) / err1)
w = [(1 / n) * math.exp(alpha1 if i in fallados else -alpha1) for i in range(n)]
total = sum(w)
w_norm = [wi / total for wi in w]
print(f"err1={err1}  alpha1={alpha1:.3f}")
print("pesos normalizados:", [round(x, 4) for x in w_norm], " suma:", round(sum(w_norm), 6))


## Reflexión

1. El laboratorio selecciona UN umbral; un ensemble de stumps combinaría varios. Con los
   5 candidatos que imprime el laboratorio y sus accuracies, ¿el voto de mayoría de los
   5 superaría al mejor individual? ¿Qué condición sobre sus errores lo decide?
2. ¿Por qué el error OOB del random forest es casi una cross-validation gratuita y qué
   fracción de árboles participa en la predicción OOB de cada ejemplo?
3. Con un 10 % de etiquetas erróneas en train, ¿esperarías más daño en random forest o en
   AdaBoost? Justifica con el mecanismo de re-ponderación.
